In [1]:
# Load environment variables from .env file
from dotenv import load_dotenv

load_dotenv()

True

In [ ]:
# Create an API client
from anthropic import Anthropic

client = Anthropic()
model = "claude-haiku-4-5" # Old model, supporting prefill

In [16]:
# Helper functions
def add_user_message(messages: list[dict], text: str):
    messages.append({"role": "user", "content": text})

def add_assistant_message(messages: list[dict], text: str):
    messages.append({"role": "assistant", "content": text})    


def chat(messages: list[dict], 
         system_prompt:str|None=None, 
         temperature:float=1.0,
         stop_sequences:list[str]|None=None) -> str:
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
    }

    if system_prompt:
        params["system"] = system_prompt

    if stop_sequences:
        params["stop_sequences"] = stop_sequences
        params["thinking"] = {"type": "disabled"} # Thinking is not supported when using assistant message prefill

    response = client.messages.create(**params)
    # print(response)
    return response.content[0].text   

In [74]:
import json


def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
        "format": "python|json|regex",
        "solution_criteria": "Description of how the solution will be graded, including edge cases to consider"
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""
    messages =[]
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)

In [75]:
dataset = generate_dataset()
dataset

[{'task': "Write a Python function that takes an AWS ARN string and extracts the resource type. For example, 'arn:aws:ec2:us-east-1:123456789012:instance/i-1234567890abcdef0' should return 'instance'.",
  'format': 'python',
  'solution_criteria': 'Function should correctly parse standard AWS ARN format (arn:partition:service:region:account-id:resource-type/resource-id or arn:partition:service:region:account-id:resource-type:resource-id). Should handle ARNs with colons and slashes as separators. Should handle edge cases like ARNs without resource IDs, ARNs with multiple colons in resource section. Should return None or raise appropriate error for invalid ARNs.'},
 {'task': "Create a JSON object representing an AWS IAM policy that allows read-only access to a specific S3 bucket named 'company-logs'. The policy should allow ListBucket and GetObject actions only.",
  'format': 'json',
  'solution_criteria': "JSON must be valid IAM policy syntax with Version, Statement array. Statement sho

In [76]:
with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)

In [77]:
from pydantic import BaseModel

class TestUseCase(BaseModel):
    task: str
    format: str
    solution_criteria: str

In [58]:
def run_prompt(test_case: TestUseCase) -> str:
    """Merges the prompt with the test case input, then returns the result"""
    prompt = f"""
    Please solve the following test:

    {test_case.task}

    * Respond with only Python, JSON or a plain Regex
    * Do not add any comments or commentary or explanations
    """

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```code")
    return chat(messages, stop_sequences=["```"])
    

In [78]:
def grade_by_model(test_case: TestUseCase, output: str) -> dict:
    """Grades the response using the model. Returns a grade of "good" or "bad"."""
    prompt = f"""
    You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution

    Original task:
    <task>
    {test_case.task}
    </task>

    Solution to evaluate:
    <solution>
    {output}
    </solution>

    Solution evaluation criteria:
    <solution_criteria>
    {test_case.solution_criteria}
    </solution_criteria>

    
    # Output Format
    Provide your evaluation as a structured JSON object with the following fields, in this specific format:
    - "strengths": An array of 1-3 key strengths
    - "weaknesses": An array of 1-3 key areas for improvement
    - "reasoning": A concise explanation of your evaluation
    - "score": A number between 1-10

    Respond only with JSON. Keep your response concise and direct.
    Examples response shape:
    {{
        "strengths": [],
        "weaknesses": [],
        "reasoning": string,
        "score": number
    }}
    """

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text)

In [79]:
grade_by_model(
    TestUseCase(
        task="Write a Python function that takes a list of numbers and returns the sum of the even numbers.", 
        format="python",
        solution_criteria="The function should correctly identify even numbers and sum them. Edge cases to consider include an empty list, a list with no even numbers, and a list with negative even numbers."), 
    "```python\ndef sum_even_numbers(numbers):\n    return sum(num for num in numbers if num % 2 == 0)\n```")

{'strengths': ['Concise and Pythonic implementation using generator expression with sum()',
  'Correctly uses modulo operator to identify even numbers (num % 2 == 0)',
  'Handles all specified edge cases correctly: empty lists return 0, lists without evens return 0, and negative evens are properly summed'],
 'weaknesses': ['Missing input validation - no type checking for the input parameter or list elements',
  'No docstring to document function purpose, parameters, and return value',
  'Could raise unclear errors if non-numeric values are in the list'],
 'reasoning': 'The solution is functionally correct and elegant, successfully solving the core problem with clean, idiomatic Python. It handles all edge cases mentioned in the criteria without additional code since sum() naturally returns 0 for empty sequences and the modulo operator works correctly for negative numbers. However, it lacks defensive programming practices like input validation and documentation that would make it product

In [63]:
import re
import ast 

def validate_json(text:str) -> int:
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0
    
def validate_python(text:str) -> int:
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0
    
def validate_regex(text:str) -> int:
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0
    
def grade_syntax(response: str, test_case: TestUseCase) -> int:
    format = test_case.format
    if format == "python":
        return validate_python(response)
    elif format == "json":
        return validate_json(response)
    elif format == "regex":
        return validate_regex(response)
    else:
        raise ValueError("Unknown task type")

In [80]:
grade_syntax(
    "def sum_even_numbers(numbers):\n    return sum(num for num in numbers if num % 2 == 0)", 
    TestUseCase(
        task="Write a Python function that takes a list of numbers and returns the sum of the even numbers.", 
        format="python",
        solution_criteria="The function should correctly identify even numbers and sum them."),)

10

In [60]:
def run_test_case(test_case: TestUseCase):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)

    # TODO - Grading
    model_grade = grade_by_model(test_case, output)
    model_score: float = model_grade["score"]
    reasoning: str = model_grade["reasoning"]

    syntax_score = grade_syntax(output, test_case)    

    score = (model_score + syntax_score) / 2

    return {
        "output": output,
        "score": score,
        "test_case": test_case.model_dump(),
        "reasoning": reasoning,
    }

In [61]:
def run_eval(dataset: list):
    """Loads the dataset and calls run_test_case for each one"""
    results = []
    for test_case in dataset:
        result = run_test_case(TestUseCase(**test_case))
        results.append(result)
    
    return results

In [81]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)


results = run_eval(dataset)

In [82]:
print(json.dumps(results, indent=2))

[
  {
    "output": "\nimport re\n\ndef extract_resource_type(arn):\n    match = re.search(r':([^/:]+)(?:/|$)', arn.split(':', 5)[5])\n    return match.group(1) if match else None\n",
    "score": 8.0,
    "test_case": {
      "task": "Write a Python function that takes an AWS ARN string and extracts the resource type. For example, 'arn:aws:ec2:us-east-1:123456789012:instance/i-1234567890abcdef0' should return 'instance'.",
      "format": "python",
      "solution_criteria": "Function should correctly parse standard AWS ARN format (arn:partition:service:region:account-id:resource-type/resource-id or arn:partition:service:region:account-id:resource-type:resource-id). Should handle ARNs with colons and slashes as separators. Should handle edge cases like ARNs without resource IDs, ARNs with multiple colons in resource section. Should return None or raise appropriate error for invalid ARNs."
    },
    "reasoning": "The solution demonstrates good understanding of ARN structure and correc

In [83]:

from statistics import mean

average_score = mean(result["score"] for result in results)
print(f"Average score: {average_score}")

Average score: 8.333333333333334
